In [0]:
## get review data and business metadata
!wget -O meta-California.json.gz "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/meta-California.json.gz"
!wget -O rating-California.csv.gz "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/rating-California.csv.gz"

In [0]:
## subset data for San Diego + Italian restaurant from business metadata
## return count of unique restaurants

import re

## get only businesses in San Diego
city_pattern = re.compile(r",\s*San Diego,\s*CA\b", re.IGNORECASE)

## filter for italian restaurants
TARGET_RESTAURANT_TYPE = "italian"

restaurant_lookup = {}

for business in read_json_gz(META_FILE):
    gmap_id = business.get("gmap_id")
    name = business.get("name")
    address = business.get("address") or ""
    categories = business.get("category") or []

    category_text = " ".join(categories) if isinstance(categories, list) else str(categories)
    category_text_lower = category_text.lower()

    is_san_diego_city = city_pattern.search(address)
    is_restaurant = "restaurant" in category_text_lower
    is_target_type = TARGET_RESTAURANT_TYPE in category_text_lower

    if gmap_id and is_san_diego_city and is_restaurant and is_target_type:
        restaurant_lookup[gmap_id] = {
            "business_name": name,
            "address": address,
            "categories": category_text,
            "latitude": business.get("latitude"),
            "longitude": business.get("longitude"),
        }

print(f"San Diego {TARGET_RESTAURANT_TYPE} restaurants found: {len(restaurant_lookup):,}")

for gmap_id, info in list(restaurant_lookup.items())[:20]:
    print("\nName:", info["business_name"])
    print("Address:", info["address"])
    print("Categories:", info["categories"])

In [0]:
## subset reviews based on San Diego + Italian restaurants (matched from businesses above)
## return number of restaurants + number of reviews

import gzip
import csv
from collections import Counter

italian_rating_count = 0
ratings_per_restaurant = Counter()

with gzip.open(RATING_FILE, "rt", encoding="utf-8") as f:
    reader = csv.reader(f)

    for i, row in enumerate(reader, start=1):
        if len(row) < 4:
            continue

        # ratings-only order is:
        # business, user, rating, timestamp
        gmap_id, user_id, rating, timestamp = row[:4]

        if gmap_id in restaurant_lookup:
            italian_rating_count += 1
            ratings_per_restaurant[gmap_id] += 1

        if i % 1_000_000 == 0:
            print(f"Processed {i:,} ratings... found {italian_rating_count:,} Italian restaurant ratings")

print(f"\nTotal Italian restaurants: {len(restaurant_lookup):,}")
print(f"Total ratings/reviews for those restaurants: {italian_rating_count:,}")